In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

In [3]:
DATASET_NAME = 'distilkaggle'
# DATASET_NAME = 'pmbf'

In [4]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /home/haotian/scs/distilkaggle
making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /home/haotian/scs/distilkaggle/dfgtree
making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /home/haotian/scs/distilkaggle/logs/full
making dirs: /home/haotian/scs/distilkaggle/models/full
making dirs: /home/haotian/scs/distilkaggle/ipynb


Connect to database.

In [5]:
db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


## Get All Candidate Notebooks

In [6]:
ids = sorted(list(
    map(
        lambda x: x["_id"],
        db.notebooksegments.find(
            {"prompted": True, "segments.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
            {"_id": 1},
        ),
    )
))
len(ids), ids[:3]

(292563,
 [ObjectId('66f3ce8e975730f2dded4f33'),
  ObjectId('66f3ce8e975730f2dded4f35'),
  ObjectId('66f3ce8e975730f2dded4f39')])

In [7]:
X_train_ids, X_test_ids = train_test_split(ids, test_size=0.2, random_state=42)
len(X_train_ids), len(X_test_ids)

(234050, 58513)

To ensure fair comparison, we use those in the test partition, i.e., their IDs in `X_test_ids`.

### Single Sample

I have already materialized all information regarding the ground truth (the code for each segment, the number of AST children, and the segment ends).

In [8]:
nb = database.get_notebooks(db, include_segments=True).next()
nb['encoding']

{'func_defs': [],
 'segments': [{'code': 'a = 2\na\nb = a * 2\nprint(b)\nc = b * 3\nprint(c)',
   'n_ast_children': 6},
  {'code': 'd = c ** 2', 'n_ast_children': 1}],
 'segment_ends': [5, 6]}

## Sample 1,024 test notebooks

Now, we sample the number of required test notebooks.

The criterion is to uniformly sample those in the joint distribution of `n_ast_children` and `n_segments`. In other words, we want to cover different patterns of these two properties as much as possible.

First, we put all test notebooks into a number of bins.

In [9]:
def get_dropping_bin(n_ast_children: int, n_segments: int):
    """n_ast_children range: 1-256, n_segments range: 4-35 (Manually select)"""
    return (n_ast_children - 1) // 32, n_segments // 4 - 1

print(get_dropping_bin(1, 4))
print(get_dropping_bin(32, 7))
print(get_dropping_bin(33, 8))
print(get_dropping_bin(64, 12))
print(get_dropping_bin(256, 35))

(0, 0)
(0, 0)
(1, 1)
(1, 2)
(7, 7)


In [10]:
bins = [[[] for _ in range(8)] for _ in range(8)]
pprint(bins)

[[[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []]]


In [11]:
pbar = tqdm(db.notebooksegments.find({"_id": {"$in": X_test_ids}, "segment_ends.3": {"$exists": True}}), total=len(X_test_ids))
n_available = 0
for nb_data in pbar:
    n_ast_children = nb_data['n_ast_children_of_segments']
    n_segments = len(nb_data['segment_ends'])

    if n_ast_children > 256 or n_segments < 4 or n_segments > 35:
        continue
        
    n_available += 1
    x, y = get_dropping_bin(n_ast_children, n_segments)
    pbar.set_postfix({"x": x, "y": y, "n": n_available})
    bins[x][y].append(nb_data)

 98%|███████████████████████████████████████ | 57210/58513 [01:40<00:02, 568.52it/s, x=0, y=0, n=55987]


Let's see the distribution...

In [12]:
[[len(bins[x][y]) for y in range(8)] for x in range(8)]

[[11263, 4300, 1589, 735, 27, 2, 1, 3],
 [6669, 4984, 2697, 988, 335, 150, 53, 58],
 [2565, 2841, 2163, 1278, 628, 351, 160, 65],
 [926, 1181, 1200, 876, 603, 396, 200, 114],
 [363, 553, 576, 542, 475, 276, 213, 116],
 [217, 266, 309, 283, 246, 226, 150, 138],
 [100, 120, 154, 173, 162, 135, 103, 85],
 [57, 59, 87, 94, 90, 88, 71, 59]]

Seems some setup cannot get enough (16) samples. We can borrow adjacent setups whenever possible.

Moreover, there is another consideration here. If two notebooks have the exact same segment ends, it may mean that they have very similar semantics (although not strictly true. But we have too many notebooks to actually analyze the code here). So we may refuse to add a notebook having the same segment ends of any of the existing notebooks in the bin.

To solve this, we can combine backtracking algorithm with recursive borrowing logic to get our final selected notebooks.

In [13]:
selected_nbs = [[[] for _ in range(8)] for _ in range(8)]
pprint(selected_nbs)

[[[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], []]]


In [14]:
n_skips = [[0 for _ in range(8)] for _ in range(8)]
pprint(n_skips)

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]


In [15]:
def select_samples_into_bins():
    N_NBS_IN_BIN = 16

    def backtrack(x: int, y: int, into_x: int, into_y: int, n_skip: int):
        nbs = bins[x][y][n_skip:]
        print(x, y, into_x, into_y, n_skip)
        
        for nb in nbs:
            has_same_segment_ends = False
            for snb in selected_nbs[x][y]:
                if nb['encoding']['segment_ends'] == snb['encoding']['segment_ends']:
                    has_same_segment_ends = True
                    continue
                    
            if has_same_segment_ends:
                print(f"WARN  find same segment ends: {snb['encoding']['segment_ends']}. skipped")
                continue

            selected_nbs[into_x][into_y].append(nb)
            if into_x != x or into_y != y:
                n_skips[x][y] += 1

            if len(selected_nbs[into_x][into_y]) >= N_NBS_IN_BIN:
                break

        if len(selected_nbs[into_x][into_y]) >= N_NBS_IN_BIN:
            return

        # borrow
        backtrack(x + 1, y, x, y, n_skips[x + 1][y])

    for y in range(8):
        for x in range(8):
            backtrack(x, y, x, y, n_skips[x][y])

In [16]:
select_samples_into_bins()

0 0 0 0 0
1 0 1 0 0
2 0 2 0 0
WARN  find same segment ends: [-1, 32, 54, 69, 96]. skipped
3 0 3 0 0
WARN  find same segment ends: [-1, 2, 22, 46, 55, 73, 116]. skipped
4 0 4 0 0
5 0 5 0 0
6 0 6 0 0
7 0 7 0 0
0 1 0 1 0
WARN  find same segment ends: [1, 9, 10, 13, 14, 15, 16, 17, 19]. skipped
WARN  find same segment ends: [0, 1, 2, 8, 9, 17, 20, 21, 22, 24, 30]. skipped
1 1 1 1 0
WARN  find same segment ends: [6, 8, 13, 21, 22, 24, 30, 36]. skipped
2 1 2 1 0
WARN  find same segment ends: [1, 3, 13, 18, 31, 44, 66, 96]. skipped
WARN  find same segment ends: [1, 3, 13, 18, 31, 44, 66, 96]. skipped
3 1 3 1 0
4 1 4 1 0
5 1 5 1 0
6 1 6 1 0
7 1 7 1 0
0 2 0 2 0
1 2 1 2 0
2 2 2 2 0
WARN  find same segment ends: [1, 9, 11, 18, 26, 31, 36, 48, 51, 54, 59, 60, 61, 66]. skipped
WARN  find same segment ends: [1, 9, 11, 18, 26, 31, 36, 48, 52, 58, 59, 65]. skipped
WARN  find same segment ends: [5, 17, 21, 26, 32, 38, 42, 50, 59, 69, 75, 81, 86]. skipped
WARN  find same segment ends: [4, 5, 16, 22, 28,

In [17]:
[[len(selected_nbs[x][y]) for y in range(8)] for x in range(8)]

[[16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 16, 16]]

In [18]:
[[n_skips[x][y] for y in range(8)] for x in range(8)]

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 14, 15, 14],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

In [19]:
def flatten(container):
    for i in container:
        if isinstance(i, (list,tuple)):
            for j in flatten(i):
                yield j
        else:
            yield i

In [27]:
result = flatten(selected_nbs)
selected_ids = [str(r['_id']) for r in result]
len(selected_ids), selected_ids[0]

(1024, '66f3ce8e975730f2dded4f33')

In [29]:
for id in selected_ids:
    db.notebook.update_one({"_id": ObjectId(id)}, {"$set": {"selected": True}})
    db.notebooksegments.update_one({"_id": ObjectId(id)}, {"$set": {"selected": True}})

In [20]:
result = [r['encoding'] for r in result]
result[:3]

[{'func_defs': [],
  'segments': [{'code': '\n\n', 'n_ast_children': 0},
   {'code': 'x = 11\ny = 3\nz = (y*(x//y))%y\n', 'n_ast_children': 3},
   {'code': '', 'n_ast_children': 0},
   {'code': 'x = random.randint(0,100)', 'n_ast_children': 1},
   {'code': '\n', 'n_ast_children': 0},
   {'code': "seq = 'reverse me!'", 'n_ast_children': 1},
   {'code': 'seq = [5,3,7,3,56,8,0,6,2]', 'n_ast_children': 1},
   {'code': 'seq = [5,3,7,3,56,8,0,6,2]', 'n_ast_children': 1},
   {'code': '', 'n_ast_children': 0},
   {'code': 's = "Lorem ipsum dolor sit amet, consectetur adipiscing elit."\n\n',
    'n_ast_children': 1},
   {'code': '', 'n_ast_children': 0},
   {'code': '', 'n_ast_children': 0},
   {'code': '', 'n_ast_children': 0},
   {'code': '\n', 'n_ast_children': 0}],
  'segment_ends': [-1, 2, 2, 3, 3, 4, 5, 6, 6, 7, 7, 7, 7, 7]},
 {'func_defs': [{'name': 'test_string',
    'code': 'def test_string():\n    s1=String("Hello")\n    s2=String("World")\n    print(s1.check("el"), s2.check("el"))\n 

Then, we build test cases that input into LLMs.

In [21]:
def make_groundtruth(encoding: dict):
    func_defs = []
    code_lines = []
    segment_ends_ast = []
    
    for fdef in encoding['func_defs']:
        func_defs.append(fdef['code'])

    for code in encoding['segments']:
        code_lines.extend([l.rstrip() for l in code['code'].split("\n") if l.rstrip()])

    [segment_ends_ast.append(x) for x in encoding['segment_ends'] if x not in segment_ends_ast and x >= 0]

    return {
        "func_defs": func_defs,
        "code_lines": code_lines,
        "segment_ends_ast": segment_ends_ast
    }

In [22]:
make_groundtruth(result[1])

{'func_defs': ['def test_string():\n    s1=String("Hello")\n    s2=String("World")\n    print(s1.check("el"), s2.check("el"))\n    print("There are %u String instances" % String.counter)\n    del s2\n    print("Now: %u String instances" % String.counter)\n    s1=1\n    print("And now: %u" % String.counter)',
  'def test_animal():\n    a = Animal(27.23,1)\n    f = Fish(3.141,2)\n    c = Cat(2.718,3)\n    pets = {\n        \'abstract\': a,\n        \'neo\': f,\n        \'snowball\': c\n        }\n    for p in pets:\n        pets[p].speak()\n        pets[p].move(2)\n    c.eat(a,pets)\n    c.eat(f,pets)\n    try:\n        pets[\'neo\'].speak()\n    except(KeyError):\n        print ("Sorry, the fish is gone")\n    f.speak()',
  'def test_ifneuron():\n    n = IFNeuron()\n    n.setIext(1)\n    n.run(1000)\n    n.plot()',
  'def test_network():\n    nn = NeuralNetwork(100)\n    nn.simulate(10000)\n    nn.neuron(0).plot()\n    nn.neuron(1).plot()',
  'def test_zoo():\n    z1=Zoo()\n    z1.add( 

In [23]:
ground_truth = [make_groundtruth(enc) for enc in result]
len(ground_truth), ground_truth[:3]

(1024,
 [{'func_defs': [],
   'code_lines': ['x = 11',
    'y = 3',
    'z = (y*(x//y))%y',
    'x = random.randint(0,100)',
    "seq = 'reverse me!'",
    'seq = [5,3,7,3,56,8,0,6,2]',
    'seq = [5,3,7,3,56,8,0,6,2]',
    's = "Lorem ipsum dolor sit amet, consectetur adipiscing elit."'],
   'segment_ends_ast': [2, 3, 4, 5, 6, 7]},
  {'func_defs': ['def test_string():\n    s1=String("Hello")\n    s2=String("World")\n    print(s1.check("el"), s2.check("el"))\n    print("There are %u String instances" % String.counter)\n    del s2\n    print("Now: %u String instances" % String.counter)\n    s1=1\n    print("And now: %u" % String.counter)',
    'def test_animal():\n    a = Animal(27.23,1)\n    f = Fish(3.141,2)\n    c = Cat(2.718,3)\n    pets = {\n        \'abstract\': a,\n        \'neo\': f,\n        \'snowball\': c\n        }\n    for p in pets:\n        pets[p].speak()\n        pets[p].move(2)\n    c.eat(a,pets)\n    c.eat(f,pets)\n    try:\n        pets[\'neo\'].speak()\n    except(K

Finally, we dump the selected notebooks.

In [24]:
fileop.write_json(ground_truth, os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))

### Find New Notebooks to Run

In [26]:
ground_truth_someempty = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth-someempty.json"))
len(ground_truth_someempty)

1024

In [31]:
new_test_samples = [x for x in ground_truth if x not in ground_truth_someempty]
len(new_test_samples)

120

In [32]:
fileop.write_json(new_test_samples, os.path.join(DATASET_NAME, f"{DATASET_NAME}_new.json"))